In [1]:
# ===== Cell 1：导入库并读取 CSV =====

from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

file_path = output_dir / "final_merged_0509_cts.csv"
fsfp_path = output_dir / "fsfp_label.csv"

# 读取数据
df = pd.read_csv(file_path, low_memory=False, encoding='utf-8', dtype={"code": str, "week": str})
df["code"] = df["code"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(6)
df["week"] = df["week"].astype(str).str.replace(".0", "", regex=False).str.strip().str.zfill(6)

fsfp_current = pd.read_csv(fsfp_path, usecols=["code", "week", "FSFP"], dtype={"code": str, "week": str})
fsfp_current["code"] = fsfp_current["code"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(6)
fsfp_current["week"] = fsfp_current["week"].astype(str).str.replace(".0", "", regex=False).str.strip().str.zfill(6)

df = df.merge(fsfp_current, on=["code", "week"], how="left")

print("CSV 读取成功。")
print(f"数据形状：{df.shape[0]} 行 × {df.shape[1]} 列")
unique_count = df.iloc[:, 0].nunique()
print(f"不重复的股票个数为: {unique_count}")

CSV 读取成功。
数据形状：1743672 行 × 153 列
不重复的股票个数为: 4643


In [2]:
# ============================================================
# Step 1. 基础数据审计 + 构造年份 / 季度辅助字段
# ============================================================

# 1. 核心列名
stock_col = "code"
week_col = "week"
fsfp_col = "FSFP_next"

required_cols = [stock_col, week_col, fsfp_col]
missing_required_cols = [col for col in required_cols if col not in df.columns]
if missing_required_cols:
    raise ValueError(f"缺少必要字段: {missing_required_cols}")

print("Step 1: 基础数据审计")
print("=" * 80)
print(f"股票代码列: {stock_col}")
print(f"周频时间列: {week_col}")
print(f"标签列: {fsfp_col}")
print(f"当前数据形状: {df.shape}")

# ------------------------------------------------------------
# 2. 检查 FSFP 分布
# ------------------------------------------------------------

print("\nFSFP 分布：")
fsfp_distribution = df[fsfp_col].value_counts(dropna=False).sort_index()
display(fsfp_distribution)


# ------------------------------------------------------------
# 3. 缺失值概览
# ------------------------------------------------------------

missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_rate": df.isna().mean().values
}).sort_values("missing_rate", ascending=False)

print("\n缺失值最多的前 20 个变量：")
display(missing_summary.head(20))

# ------------------------------------------------------------
# 4. 将 week 列解析为年份、周数和季度
# ------------------------------------------------------------
# week 格式固定为 YYYYWW，例如：
# 201501 表示 2015 年第 1 周

week_str = df[week_col].astype(str).str.strip()
invalid_week_mask = ~week_str.str.fullmatch(r"\d{6}", na=False)
df["Year"] = week_str.str[:4].astype(int)
df["week_num"] = week_str.str[4:6].astype(int)

Step 1: 基础数据审计
股票代码列: code
周频时间列: week
标签列: FSFP_next
当前数据形状: (1743672, 153)

FSFP 分布：
FSFP_next
0    784764
1    488692
2    470216
Name: count, dtype: int64

缺失值最多的前 20 个变量：
                     column  missing_count  missing_rate
22                gov__NSAO        1742142      0.999123
20                 gov__IHR        1591998      0.913015
26                 mkt__Vol        1427076      0.818431
25            mkt__Turnover        1422779      0.815967
129  peer_gov__NSAO-NSAO_均值        1238082      0.710043
128       peer_gov__NSAO_均值        1238082      0.710043
127          peer_gov__NSAO        1238082      0.710043
68         peer_fin__FSR_均值         617405      0.354083
14                 fin__FSR         617405      0.354083
67            peer_fin__FSR         617405      0.354083
69     peer_fin__FSR-FSR_均值         617405      0.354083
18                gov__3yMC         439083      0.251815
107       peer_gov__3yMC_均值         375845      0.215548
106          peer_gov__3yM

In [3]:
columns_to_drop = [
    "fin__FSR",
    "peer_fin__FSR-FSR_均值",
    "gov__3yMC",
    "peer_gov__3yMC-3yMC_均值",
    "peer_fin__EIR-EIR_均值",
    "fin__EIR",
]
existing_drop_cols = [col for col in columns_to_drop if col in df.columns]
missing_drop_cols = [col for col in columns_to_drop if col not in df.columns]
df = df.drop(columns=columns_to_drop, errors="ignore")
print("已删除的问题列：", existing_drop_cols)
print("未在当前数据中找到的待删列：", missing_drop_cols)
# 识别“整数值变量列”：
# 包括本身是 int 类型的列，以及虽然是 float 类型但非空值全都是整数的列
integer_cols = []

for col in df.columns:
    s = df[col]
    # 只判断数值型列
    if pd.api.types.is_numeric_dtype(s):
        non_na = s.dropna()
        # 空列先不处理，避免误判
        if len(non_na) == 0:
            continue
        # 判断非空值是否全部为整数
        if np.all(np.isclose(non_na, np.round(non_na))):
            integer_cols.append(col)
df[integer_cols] = df[integer_cols].fillna(0)
print("识别出的整数值变量列：")
print(integer_cols)
output_path = output_dir / "final_merged_0509_cts_cleaned_FSFP.csv"
df.to_csv(output_path, index=False, encoding='utf-8')
print("清洗后文件已保存：", output_path)

已删除的问题列： ['fin__FSR', 'peer_fin__FSR-FSR_均值', 'gov__3yMC', 'peer_gov__3yMC-3yMC_均值', 'peer_fin__EIR-EIR_均值', 'fin__EIR']
未在当前数据中找到的待删列： []
识别出的整数值变量列：
['FSFP_next', 'fin__2yLoss', 'fin__NCFO', 'gov__Board', 'gov__NSAO', 'mkt__MrkVal', 'sent__RepoSent', 'sent__GubaSent', 'sent__NewsSent', 'peer_fin__NCFO', 'peer_fin__2yLoss', 'peer_mkt__WeekID', 'peer_mkt__WeekID_均值', 'peer_mkt__MrkVal', 'peer_mkt__MrkVal_均值', 'peer_mkt__MrkVal-MrkVal_均值', 'peer_gov__3yMC', 'peer_gov__Board', 'peer_gov__DUAL', 'peer_gov__SOE', 'peer_gov__NSAO', 'peer_gov__3yAFC', 'peer_gov__3yBC', 'peer_gov__Top4Aud', 'peer_sent__GubaSent', 'peer_sent__NewsSent', 'peer_sent__RepoSent', 'FSFP', 'Year', 'week_num']
清洗后文件已保存： output\final_merged_0509_cts_cleaned_FSFP.csv
